# E3: Regiones, semillas y ablaciones
**Proyecto:** Clasificación de cobertura del suelo con DINOv2 (EuroSAT RGB) · 1INF52 Deep Learning 2026-2  
**Pregunta de investigación:** ¿Añadir adaptadores LoRA mejora la clasificación en regiones geográficas nuevas frente a usar DINOv2 congelado?  
**Fase C:** Generalización espacial por bloques disjuntos, evaluación pareada con 3 semillas (A2, A4) y ablaciones simétricas (A3, A5, A6).


> **Nota (25/11/2026):** este notebook se conserva tal como se ejecutó en Colab (20 corridas). Las tablas impresas en la sección 5 tienen dos errores del análisis original, ya corregidos en `src/eurosat_lora/analisis.py`: (1) el veredicto de A4 comparaba fracciones con umbrales en puntos y (2) C2/C4 mezclaban LoRA r8 con las ablaciones de la misma semilla. Las tablas y figuras válidas son las de `results/C*.csv` y `figures/C*.png`, generadas con `python scripts/E3_analisis.py` a partir de las corridas guardadas en `results/E3_corridas/`. Desviaciones del entrenamiento declaradas en `DESIGN_LOG.md` (D21).


In [1]:
# 0. Configuración del entorno y almacenamiento de resultados
import os
import sys

EN_COLAB = "google.colab" in sys.modules

# =========================================================================
# CONFIGURACIÓN DE ALMACENAMIENTO (Sin necesidad de vincular Google Drive)
# Por defecto se guarda localmente en el disco de Colab (/content/outputs/E3).
# Si deseas vincular tu Google Drive personal, cambia USAR_DRIVE a True.
# =========================================================================
USAR_DRIVE = False

if EN_COLAB and USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SALIDA = "/content/drive/MyDrive/E3"
        print("Google Drive conectado correctamente.")
    except Exception as e:
        print(f"No se pudo conectar Google Drive ({e}). Usando disco local de Colab.")
        SALIDA = "/content/outputs/E3"
elif EN_COLAB:
    SALIDA = "/content/outputs/E3"
    print("Guardando directamente en el almacenamiento de Colab (/content/outputs/E3).")
else:
    SALIDA = "outputs/E3"
    print(f"Entorno local: {SALIDA}")

os.makedirs(SALIDA, exist_ok=True)
MODO_PRUEBA = False  # True para prueba ultrarrápida con max_por_clase=20 y 1 época
print(f"Directorio de salida: {SALIDA} (MODO_PRUEBA={MODO_PRUEBA})")


Guardando directamente en el almacenamiento de Colab (/content/outputs/E3).
Directorio de salida: /content/outputs/E3 (MODO_PRUEBA=False)


In [2]:
# 0.1 Preparación del repositorio y paquete modular en Colab
import os
import sys

# Agregar rutas 'src' a sys.path tanto para entorno local como Colab
for ruta in [
    os.path.abspath("src"),
    "/content/deep-learning/src",
    "/content/dinov2-eurosat-lora/src",
]:
    if os.path.isdir(ruta) and ruta not in sys.path:
        sys.path.insert(0, ruta)

if EN_COLAB:
    # Si no existe el código fuente en el entorno, clonar repositorio
    if not any(os.path.isdir(p) for p in ["src/eurosat_lora", "/content/deep-learning/src/eurosat_lora", "/content/dinov2-eurosat-lora/src/eurosat_lora"]):
        print("Clonando repositorio con paquete eurosat_lora...")
        !git clone -b main https://github.com/jflxx/deep-learning.git /content/deep-learning || git clone -b main https://github.com/italo04/dinov2-eurosat-lora.git /content/dinov2-eurosat-lora
    else:
        # Si ya existe /content/deep-learning, actualizar con git pull
        if os.path.isdir("/content/deep-learning/.git"):
            !cd /content/deep-learning && git pull origin main

    # Asegurar que el directorio de trabajo y las rutas estén en sys.path
    if os.path.isdir("/content/deep-learning"):
        %cd /content/deep-learning
        if "/content/deep-learning/src" not in sys.path:
            sys.path.insert(0, "/content/deep-learning/src")
    elif os.path.isdir("/content/dinov2-eurosat-lora"):
        %cd /content/dinov2-eurosat-lora
        if "/content/dinov2-eurosat-lora/src" not in sys.path:
            sys.path.insert(0, "/content/dinov2-eurosat-lora/src")

    print("Instalando dependencias...")
    !pip install -q rasterio pyproj
    !pip install -q -e .
    print("Paquete eurosat_lora y dependencias configurados exitosamente.")


Clonando repositorio con paquete eurosat_lora...
Cloning into '/content/deep-learning'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 123 (delta 32), reused 27 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 6.43 MiB | 12.44 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/deep-learning
Instalando dependencias...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for eurosat_lora (pyproject.toml) ... done
Paquete eurosat_lora y dependencias configurados exitosamente.


## 1. Datos y partición interna (reproducibilidad de E2)

In [3]:
import os
import sys

# Auto-verificación de rutas para eurosat_lora (a prueba de reinicios de entorno en Colab)
for ruta in [
    os.path.abspath("src"),
    "/content/deep-learning/src",
    "/content/dinov2-eurosat-lora/src",
]:
    if os.path.isdir(ruta) and ruta not in sys.path:
        sys.path.insert(0, ruta)

try:
    from eurosat_lora.config import Config
    from eurosat_lora.datos import (
        descargar_eurosat_rgb,
        particion_estratificada,
        archivos_de,
    )
except (ModuleNotFoundError, ImportError):
    print("Actualizando paquete eurosat_lora en el entorno de Colab...")
    if not os.path.isdir("/content/deep-learning") and not os.path.isdir("/content/dinov2-eurosat-lora"):
        !git clone -b main https://github.com/jflxx/deep-learning.git /content/deep-learning || git clone -b main https://github.com/italo04/dinov2-eurosat-lora.git /content/dinov2-eurosat-lora
    else:
        if os.path.isdir("/content/deep-learning/.git"):
            !cd /content/deep-learning && git pull origin main
    for r in ["/content/deep-learning/src", "/content/dinov2-eurosat-lora/src", os.path.abspath("src")]:
        if os.path.isdir(r) and r not in sys.path:
            sys.path.insert(0, r)
    !pip install -q rasterio pyproj
    import importlib
    importlib.invalidate_caches()
    from eurosat_lora.config import Config
    try:
        from eurosat_lora.datos import (
            descargar_eurosat_rgb,
            particion_estratificada,
            archivos_de,
        )
    except ImportError:
        from eurosat_lora.datos import (
            preparar_eurosat_rgb as descargar_eurosat_rgb,
            particion_estratificada,
            archivos_de,
        )

import numpy as np
import pandas as pd
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder

data_dir = "data/EuroSAT_RGB"
if not os.path.exists(data_dir):
    data_dir = descargar_eurosat_rgb("data")

tf = transforms.Compose([
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
dataset = ImageFolder(data_dir, transform=tf)
clases = dataset.classes
print(f"EuroSAT RGB cargado: {len(dataset)} imágenes en {len(clases)} clases: {clases}")

# Partición interna reproducible E2 (70/15/15)
y_targets = np.array(dataset.targets)
idx_tr, idx_va, idx_te = particion_estratificada(y_targets, frac_val=0.15, frac_test=0.15, seed=42)
print(f"Partición interna: train={len(idx_tr)}, val={len(idx_va)}, test={len(idx_te)}")
assert len(idx_tr) == 18900 and len(idx_va) == 4050 and len(idx_te) == 4050

particion_interna = {
    "train": archivos_de(dataset, idx_tr),
    "val": archivos_de(dataset, idx_va),
    "test": archivos_de(dataset, idx_te),
}


EuroSAT RGB cargado: 27000 imágenes en 10 clases: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Partición interna: train=18900, val=4050, test=4050


## 2. Coordenadas y partición geográfica (bloques espaciales)

In [4]:
import json
import os

ruta_split_geo = 'splits/split_geo.json'
if not os.path.exists(ruta_split_geo):
    print(f'"{ruta_split_geo}" no encontrado en el repositorio local.')
    print('Generando partición geográfica por bloques espaciales (D17)...')
    os.makedirs('splits', exist_ok=True)
    if not os.path.exists('splits/eurosat_coords.csv'):
        print('Descargando EuroSAT MS para extraer coordenadas exactas de los GeoTIFFs...')
        !python scripts/E3_coordenadas_eurosat_ms.py --rgb-dir data/EuroSAT_RGB
    print('Calculando partición geográfica K-means...')
    !python scripts/E3_particion_geo.py

with open(ruta_split_geo, 'r', encoding='utf-8') as f:
    particion_geo = json.load(f)

print('Partición geográfica cargada exitosamente:')
print(f"  Train:        {len(particion_geo['train'])} imágenes")
print(f"  Val:          {len(particion_geo['val'])} imágenes")
print(f"  Test dentro:  {len(particion_geo['test_dentro'])} imágenes")
print(f"  Test fuera:   {len(particion_geo['test_fuera'])} imágenes")
print(f"  Regiones k={particion_geo['k']}, intento de respaldo={particion_geo['intento_respaldo']}")

particiones = {
    'interna': particion_interna,
    'geo': particion_geo,
}


"splits/split_geo.json" no encontrado en el repositorio local.
Generando partición geográfica por bloques espaciales (D17)...
Descargando EuroSAT MS para extraer coordenadas exactas de los GeoTIFFs...
Descargando EuroSAT MS desde https://zenodo.org/records/7711810/files/EuroSAT_MS.zip...
Descarga completada.
Extrayendo archivo zip...
Extracción completada.
Encontrados 27000 archivos GeoTIFF en data/EuroSAT_MS
Procesando 5000/27000...
Procesando 10000/27000...
Procesando 15000/27000...
Procesando 20000/27000...
Procesando 25000/27000...
Procesando 27000/27000...
Verificando emparejamiento con 27000 archivos RGB...
Emparejamiento perfecto MS <-> RGB verificado.
Coordenadas guardadas en splits/eurosat_coords.csv (27000 registros)
Calculando partición geográfica K-means...
Cargando coordenadas desde splits/eurosat_coords.csv...
Total imágenes: 27000
Calculando partición geográfica por bloques espaciales...
Partición generada con k=15, respaldo=1
Guardado split en splits/split_geo.json
Guar

## 3. Verificación de hipótesis previas (DESIGN_LOG.md)

In [5]:
from eurosat_lora.experimentos import verificar_predicciones, PREDICCION_REQUERIDA

verificar_predicciones(PREDICCION_REQUERIDA, "DESIGN_LOG.md")
print("Verificación exitosa: Todas las predicciones de las ablaciones están registradas en DESIGN_LOG.md.")


Verificación exitosa: Todas las predicciones de las ablaciones están registradas en DESIGN_LOG.md.


## 4. Matriz de experimentos (20 corridas idempotentes)

In [6]:
from eurosat_lora.experimentos import (
    MATRIZ_E3,
    Contexto,
    ejecutar_corrida,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
amp = True if device == "cuda" else False
print(f"Dispositivo: {device} (AMP={amp})")

cfg_base = Config(
    modelo_id="facebook/dinov2-small",
    img_size=224,
    batch=64,
    epocas=1 if MODO_PRUEBA else 8,
    lr=1e-3,
    weight_decay=1e-4,
    max_por_clase=20 if MODO_PRUEBA else None,
    num_workers=2,
)

ctx = Contexto(
    dataset=dataset,
    clases=clases,
    particiones=particiones,
    cfg=cfg_base,
    device=device,
    amp=amp,
)

print(f"Iniciando ejecución de {len(MATRIZ_E3)} corridas...")
for idx, corrida in enumerate(MATRIZ_E3, 1):
    print(f"[{idx:02d}/{len(MATRIZ_E3)}] {corrida.id} (Exp: {corrida.experimento})...")
    res = ejecutar_corrida(corrida, ctx, salida=SALIDA)
    print(f"      Completada en {res['segundos']} s.")


Dispositivo: cuda (AMP=True)
Iniciando ejecución de 20 corridas...
[01/20] interna_base_r8_b10-11_q+v_s42 (Exp: A2)...


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

... (29 outputs left out)

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

      Completada en 535.0 s.
[20/20] geo_lora_r8_b10-11_q_s42 (Exp: A6g)...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

      Completada en 500.2 s.


## 5. Análisis de resultados, tablas de síntesis y veredicto de A4

In [7]:
from eurosat_lora.analisis import (
    cargar_corridas,
    tabla_semillas,
    tabla_mcnemar,
    tabla_fase_c,
    veredicto_a4,
    tabla_ablaciones,
)

corridas_dict = cargar_corridas(SALIDA)
os.makedirs("results", exist_ok=True)

# C2: Variación entre semillas
df_semillas = tabla_semillas(corridas_dict)
df_semillas.to_csv("results/C2_semillas.csv", index=False)
print("=== Tabla C2 (Semillas) ===")
display(df_semillas) if "display" in dir() else print(df_semillas)

# C3: McNemar exacto
df_mcnemar = tabla_mcnemar(SALIDA, corridas_dict)
df_mcnemar.to_csv("results/C3_mcnemar_exacto.csv", index=False)
print("\n=== Tabla C3 (McNemar exacto) ===")
display(df_mcnemar) if "display" in dir() else print(df_mcnemar)

# C4: Fase C (Geográfica) y veredicto A4
df_fase_c = tabla_fase_c(corridas_dict)
df_fase_c.to_csv("results/C4_fase_c.csv", index=False)
print("\n=== Tabla C4 (Fase C) ===")
display(df_fase_c) if "display" in dir() else print(df_fase_c)

resultado_a4 = veredicto_a4(df_fase_c)
print(f"\n=======================================================")
print(f"VEREDICTO A4: {resultado_a4['veredicto'].upper()}")
print(f"Signo consistente: {resultado_a4['signo_consistente']}")
print(f"Caída media base: {resultado_a4['caida_base_media']:.2f} pts | LoRA: {resultado_a4['caida_lora_media']:.2f} pts")
print(f"Ventaja media LoRA fuera: {resultado_a4['ventaja_fuera_media']:.2f} pts")
print(f"=======================================================\n")

# C7: Ablaciones
df_abl = tabla_ablaciones(corridas_dict)
df_abl.to_csv("results/C7_ablaciones.csv", index=False)
print("=== Tabla C7 (Ablaciones) ===")
display(df_abl) if "display" in dir() else print(df_abl)


=== Tabla C2 (Semillas) ===
   particion condicion  semilla     conjunto  macro_f1
0        geo      base        0  test_dentro  0.961549
1        geo      base        0   test_fuera  0.884639
2        geo      base        1  test_dentro  0.965901
3        geo      base        1   test_fuera  0.901959
4        geo      base       42  test_dentro  0.964570
5        geo      base       42   test_fuera  0.890871
6        geo      lora        0  test_dentro  0.974886
7        geo      lora        0   test_fuera  0.918169
8        geo      lora        1  test_dentro  0.973567
9        geo      lora        1   test_fuera  0.917534
10       geo      lora       42  test_dentro  0.973610
11       geo      lora       42   test_fuera  0.910342
12   interna      base        0         test  0.958696
13   interna      base        1         test  0.955039
14   interna      base       42         test  0.955933
15   interna      lora        0         test  0.971907
16   interna      lora        1      

## 6. Diagnósticos y análisis de representaciones

In [8]:
from eurosat_lora.diagnosticos import norma_tokens, remedir_fp16_b1
from eurosat_lora.modelo import cargar_dinov2

m_diag = cargar_dinov2(cfg_base.modelo_id, device=device)

# C9: Norma de tokens en bloque 12
x_muestra = torch.stack([dataset[i][0] for i in range(min(32, len(dataset)))]).to(device)
normas = norma_tokens(m_diag, x_muestra)
df_normas = pd.DataFrame(normas)
df_normas.to_csv("results/C9_norma_tokens.csv", index=False)
print(f"Norma de tokens calculada (forma {normas.shape}) -> results/C9_norma_tokens.csv")

# C10: Latencia fp16 batch 1 (si hay GPU)
if device == "cuda":
    df_lat = remedir_fp16_b1(m_diag, device=device)
    df_lat.to_csv("results/C10_latencia_fp16_b1.csv", index=False)
    print("Latencia fp16 b1 guardada en results/C10_latencia_fp16_b1.csv")


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Norma de tokens calculada (forma (32, 257)) -> results/C9_norma_tokens.csv
Latencia fp16 b1 guardada en results/C10_latencia_fp16_b1.csv


## 7. Exportación de checkpoints y artefactos

In [9]:
import shutil

os.makedirs("checkpoints/geo_s42", exist_ok=True)
ckpt_origen = os.path.join(SALIDA, "geo_lora_r8_b10-11_q+v_s42", "lora_y_cabeza.pt")
if os.path.exists(ckpt_origen):
    shutil.copy(ckpt_origen, "checkpoints/geo_s42/lora_y_cabeza.pt")
    print("Checkpoint geo semilla 42 copiado a checkpoints/geo_s42/lora_y_cabeza.pt")

# Generar zip con todos los resultados listo para descargar
if EN_COLAB:
    zip_salida = "/content/resultados_E3"
    shutil.make_archive(zip_salida, "zip", SALIDA)
    print(f"\n Archivo comprimido creado en: {zip_salida}.zip")
    print("Puedes descargarlo haciendo clic derecho en el panel izquierdo de archivos de Colab o ejecutando:")
    print("from google.colab import files; files.download('/content/resultados_E3.zip')")

print("Proceso completado exitosamente.")


Checkpoint geo semilla 42 copiado a checkpoints/geo_s42/lora_y_cabeza.pt

 Archivo comprimido creado en: /content/resultados_E3.zip
Puedes descargarlo haciendo clic derecho en el panel izquierdo de archivos de Colab o ejecutando:
from google.colab import files; files.download('/content/resultados_E3.zip')
Proceso completado exitosamente.
